# Notebook 03c v2 — CIC-IDS2017 Dirichlet calibration

**What this notebook does:**

Fits Dirichlet ODIR calibrator per model on CIC calibration set, applies to test, compares against pre-cal and hybrid Platt/isotonic (loaded from Notebook 03-CIC v2).

**CIC is the hardest test for Dirichlet:**

- U2R has only 7 calibration samples (smaller than NSL's 10 and UNSW's 253)
- Dirichlet ODIR fits a 5×5 transformation matrix; with n_calib=7 in one class, the fit is at the edge of stability
- If Dirichlet ODIR struggles anywhere, it should struggle here

**Expected output:** Dirichlet fit may issue warnings or produce unstable U2R probabilities. We report what happens honestly.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
!pip install -q dirichletcal
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator
print('✓ dirichletcal package loaded')

  Preparing metadata (setup.py) ... done
✓ dirichletcal package loaded


In [3]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.metrics import brier_score_loss
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASET = 'cic_ids2017_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PROBS_DIR = MODELS_DIR / 'probabilities'
CALIB_DIR_HYBRID = Path(REPO) / 'calibrators' / DATASET
CALIB_DIR_DIRICHLET = Path(REPO) / 'calibrators' / f'{DATASET}_dirichlet'
CALIB_DIR_DIRICHLET.mkdir(parents=True, exist_ok=True)

ECE_N_BINS = 10
print(f'Dataset: {DATASET}, Dirichlet outputs: {CALIB_DIR_DIRICHLET}')

Dataset: cic_ids2017_v2, Dirichlet outputs: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/cic_ids2017_v2_dirichlet


## 2. Load labels and class priors

In [4]:
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)

print(f'class_mappings.json keys: {list(class_info.keys())}')
mapping_5 = None
for key_candidate in ['multiclass_5', 'five_class_id_to_name', '5class', 'five_class', 'multiclass',
                       'cic_id_to_name', 'class_id_to_name', 'id_to_name']:
    if key_candidate in class_info:
        mapping_5 = class_info[key_candidate]
        print(f"Using key: '{key_candidate}'")
        break

if mapping_5 is None:
    print(f'\nFull JSON content:')
    print(json.dumps(class_info, indent=2))
    raise KeyError('No 5-class mapping found.')

INT_TO_CATEGORY = {int(k): v for k, v in mapping_5.items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'\nCalibration set: {len(y_calib_b):,}, Test set: {len(y_test_b):,}')
print(f'5-class names: {CLASS_NAMES_5}')

calib_counts = Counter(y_calib_5)
test_counts = Counter(y_test_5)
n_calib = len(y_calib_5)
n_test = len(y_test_5)

rows = []
for c in range(5):
    p_calib = calib_counts[c] / n_calib
    p_test = test_counts[c] / n_test
    rows.append({
        'Class': CLASS_NAMES_5[c],
        'n_calib': calib_counts[c],
        'p_calib (%)': round(100 * p_calib, 2),
        'n_test': test_counts[c],
        'p_test (%)': round(100 * p_test, 2),
        'p_test/p_calib': round(p_test / p_calib, 2) if p_calib > 0 else float('inf'),
    })

df = pd.DataFrame(rows)
print('\nCIC class-prior comparison')
print('=' * 90)
print(df.to_string(index=False))
print('=' * 90)

class_mappings.json keys: ['binary', 'multiclass_5', 'label_mapping', 'note']
Using key: 'multiclass_5'

Calibration set: 40,006, Test set: 40,007
5-class names: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

CIC class-prior comparison
 Class  n_calib  p_calib (%)  n_test  p_test (%)  p_test/p_calib
Normal    32152        80.37   32153       80.37             1.0
   DoS     5376        13.44    5376       13.44             1.0
 Probe     2248         5.62    2248        5.62             1.0
   R2L      223         0.56     223        0.56             1.0
   U2R        7         0.02       7        0.02             1.0


## 3. Helpers

In [5]:
def expected_calibration_error(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (probs >= lo) & (probs <= hi) if i == n_bins - 1 else (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(probs[mask].mean() - labels[mask].mean())
    return float(ece)

def fit_dirichlet(p_calib_2d, y_calib, reg_lambda=1e-3):
    cal = FullDirichletCalibrator(reg_lambda=reg_lambda, reg_mu=None)
    cal.fit(p_calib_2d, y_calib)
    return cal

def evaluate_calibrator(p_test_cal, y_test, n_classes):
    brier = {}
    ece = {}
    for c in range(n_classes):
        y_c = (y_test == c).astype(int)
        p_c = p_test_cal[:, c]
        brier[c] = float(brier_score_loss(y_c, p_c))
        ece[c] = expected_calibration_error(p_c, y_c, ECE_N_BINS)
    return brier, ece

print('✓ Helpers loaded')

✓ Helpers loaded


## 4. Calibrate all 9 models with Dirichlet

In [ ]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task}) with Dirichlet ODIR...')
    p_calib = np.load(PROBS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PROBS_DIR / f'{name}_test_proba.npy')

    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])

    n_classes = p_calib.shape[1]
    y_calib_use = y_calib_b if task == 'binary' else y_calib_5
    y_test_use = y_test_b if task == 'binary' else y_test_5

    brier_pre, ece_pre = evaluate_calibrator(p_test, y_test_use, n_classes)

    try:
        cal = fit_dirichlet(p_calib, y_calib_use)
        p_test_cal = cal.predict_proba(p_test)
        success = True
    except Exception as e:
        print(f'  ✗ Dirichlet fit failed: {e}')
        print(f'    Falling back to identity (no calibration).')
        p_test_cal = p_test.copy()
        success = False

    brier_post, ece_post = evaluate_calibrator(p_test_cal, y_test_use, n_classes)
    np.save(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy', p_test_cal)

    if task == '5class':
        for c in range(5):
            delta_b = brier_post[c] - brier_pre[c]
            arrow = '↓' if delta_b < 0 else ('↑' if delta_b > 0 else '−')
            print(f'  {CLASS_NAMES_5[c]:8s}: Brier {brier_pre[c]:.4f} → {brier_post[c]:.4f} {arrow}'
                  f'   ECE {ece_pre[c]:.4f} → {ece_post[c]:.4f}')
    else:
        delta_b = brier_post[1] - brier_pre[1]
        arrow = '↓' if delta_b < 0 else ('↑' if delta_b > 0 else '−')
        print(f'  Attack  : Brier {brier_pre[1]:.4f} → {brier_post[1]:.4f} {arrow}'
              f'   ECE {ece_pre[1]:.4f} → {ece_post[1]:.4f}')

    calibration_summary[name] = {
        'task': task,
        'method': 'dirichlet_odir' if success else 'failed_identity',
        'success': success,
        'brier_pre':  {int(k): v for k, v in brier_pre.items()},
        'brier_post': {int(k): v for k, v in brier_post.items()},
        'ece_pre':  {int(k): v for k, v in ece_pre.items()},
        'ece_post': {int(k): v for k, v in ece_post.items()},
    }

with open(CALIB_DIR_DIRICHLET / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ Done. Summary: {CALIB_DIR_DIRICHLET}/calibration_summary.json')


Calibrating rf_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.0014 → 0.0015 ↑   ECE 0.0016 → 0.0021

Calibrating xgb_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.0009 → 0.0009 ↓   ECE 0.0007 → 0.0007

Calibrating dnn_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.0208 → 0.0172 ↓   ECE 0.0198 → 0.0070

Calibrating rf_5class_smote (5class) with Dirichlet ODIR...


## 5. Three-way comparison (pre / hybrid / Dirichlet)

In [ ]:
hybrid_summary = None
hybrid_path = CALIB_DIR_HYBRID / 'calibration_summary.json'
if hybrid_path.exists():
    with open(hybrid_path) as f:
        hybrid_summary = json.load(f)
    print(f'✓ Hybrid summary found: {hybrid_path}')
else:
    print(f'⚠ Hybrid summary not found, comparison will be pre vs Dirichlet only')

rows = []
for name, task in ALL_MODELS:
    d = calibration_summary[name]
    d_pre = d['brier_pre']
    d_post = d['brier_post']

    if task == '5class':
        b_pre = np.mean([d_pre[c] for c in range(5)])
        b_dirichlet = np.mean([d_post[c] for c in range(5)])
    else:
        b_pre = d_pre[1]
        b_dirichlet = d_post[1]

    row = {
        'Model': name,
        'Task': '5-class' if task == '5class' else 'binary',
        'Brier pre':       round(b_pre,       5),
        'Brier dirichlet': round(b_dirichlet, 5),
        'Dirichlet delta': round(b_dirichlet - b_pre, 5),
    }

    if hybrid_summary is not None and name in hybrid_summary:
        h = hybrid_summary[name]
        h_post = h['brier_post']
        if task == '5class':
            b_hybrid = np.mean([h_post[str(c)] for c in range(5)])
        else:
            b_hybrid = h_post['1']
        row['Brier hybrid'] = round(b_hybrid, 5)
        methods = {'pre': b_pre, 'hybrid': b_hybrid, 'dirichlet': b_dirichlet}
    else:
        methods = {'pre': b_pre, 'dirichlet': b_dirichlet}
    row['Winner'] = min(methods, key=methods.get)
    rows.append(row)

df_cmp = pd.DataFrame(rows)
print('\nCIC v2 — Calibration comparison (macro Brier)')
print('=' * 110)
print(df_cmp.to_string(index=False))
print('=' * 110)

TABLES_DIR = Path(REPO) / 'results' / 'tables'
csv_path = TABLES_DIR / 'cic_v2_calibration_threeway.csv'
df_cmp.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

winners = df_cmp['Winner'].value_counts()
print(f'\nWinner counts: {winners.to_dict()}')

## 6. Per-class three-way comparison

In [ ]:
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    d = calibration_summary[name]

    for c in range(5):
        d_pre = d['brier_pre'][c]
        d_post = d['brier_post'][c]
        row = {
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts[c],
            'Brier pre': round(d_pre, 5),
            'Brier dirichlet': round(d_post, 5),
            'Delta': round(d_post - d_pre, 5),
        }
        if hybrid_summary is not None and name in hybrid_summary:
            h = hybrid_summary[name]
            h_post = h['brier_post'][str(c)]
            h_strategy = h['strategies'][str(c)]
            row['Brier hybrid'] = round(h_post, 5)
            row['Hybrid strategy'] = h_strategy
            methods = {'pre': d_pre, 'hybrid': h_post, 'dirichlet': d_post}
        else:
            methods = {'pre': d_pre, 'dirichlet': d_post}
        row['Winner'] = min(methods, key=methods.get)
        perclass_rows.append(row)

df_perclass = pd.DataFrame(perclass_rows)
print('CIC v2 — Per-class comparison')
print('=' * 140)
print(df_perclass.to_string(index=False))
print('=' * 140)

csv_path = TABLES_DIR / 'cic_v2_calibration_threeway_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

print(f'\nWinners by class:')
for c in range(5):
    cname = CLASS_NAMES_5[c]
    cnt = Counter(df_perclass[df_perclass['Class'] == cname]['Winner'])
    print(f'  {cname:8s}: {dict(cnt)}')

## 7. Argmax preservation check

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print('CIC v2 — Dirichlet impact on classifier predictions')
print('=' * 100)
print(f'{"Model":<22} {"% flipped":>10} {"Acc pre":>9} {"Acc post":>10} {"Δ acc":>9} {"F1m pre":>9} {"F1m post":>10} {"Δ F1m":>9}')
print('-' * 100)

for name, task in ALL_MODELS:
    if task != '5class':
        continue
    p_pre = np.load(PROBS_DIR / f'{name}_test_proba.npy')
    p_post = np.load(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy')
    pred_pre = p_pre.argmax(axis=1)
    pred_post = p_post.argmax(axis=1)
    n_flipped = (pred_pre != pred_post).sum()
    pct_flipped = 100 * n_flipped / len(pred_pre)

    acc_pre = accuracy_score(y_test_5, pred_pre)
    acc_post = accuracy_score(y_test_5, pred_post)
    f1_pre = f1_score(y_test_5, pred_pre, average='macro', zero_division=0)
    f1_post = f1_score(y_test_5, pred_post, average='macro', zero_division=0)

    print(f'{name:<22} {pct_flipped:>9.2f}% {acc_pre:>9.4f} {acc_post:>10.4f} {acc_post-acc_pre:>+9.4f} '
          f'{f1_pre:>9.4f} {f1_post:>10.4f} {f1_post-f1_pre:>+9.4f}')

## 8. Sanity checks

In [ ]:
print('Sanity checks on Dirichlet calibrated probabilities:')
print('-' * 80)
for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy')
    expected_cols = 2 if task == 'binary' else 5
    shape_ok = p_cal.shape[1] == expected_cols
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    finite_ok = np.isfinite(p_cal).all()
    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

## 9. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/03c_cic_calibration_dirichlet_v2.ipynb
!git add calibrators/cic_ids2017_v2_dirichlet/
!git add results/tables/cic_v2_calibration_threeway.csv
!git add results/tables/cic_v2_calibration_threeway_perclass.csv
!git status --short
!git commit -m 'Notebook 03c-CIC v2: Dirichlet calibration with argmax preservation check'
!git push origin main